# LazyAd — Phase 1, Spike S2: User-Supplied Product Image Input

**Source of Truth:** `Docs/Phase 1, Spike S2.md` & `Docs/PHASE 0 FEASIBILITY & REQUIREMENTS.md` (Approved v2.2)

### Objective
Verify that a user-supplied product image can be safely uploaded/loaded in Google Colab, validated, previewed inline, and prepared as a standard input (`/content/lazyad_assets/product_input.png`) for the later TripoSR 3D reconstruction spike (Spike S3).

### Supported Formats & Input Methods
- **Formats:** PNG, JPEG / JPG, WebP.
- **Methods:** Direct interactive upload widget (`google.colab.files.upload()`) or loading from a specified file path in `/content`.
- **Zero API Dependencies:** Requires **no** Google Gemini API keys, **no** billing, and **no** heavy diffusion models.

> **Strict Scope Limits:** Spike S2 is an image-input validation and preparation stage only. It does **not** install TripoSR, does **not** generate a GLB, and does **not** render video.

## Step 1: Destination Asset Directory Setup
Ensures `/content/lazyad_assets/` exists for storing inputs and intermediate artifacts.

In [ ]:
import os
import shutil

ASSET_DIR = "/content/lazyad_assets"
os.makedirs(ASSET_DIR, exist_ok=True)
TARGET_IMAGE_PATH = os.path.join(ASSET_DIR, "product_input.png")

print(f"✅ Asset directory ready: {ASSET_DIR}")
print(f"🎯 Target prepared image path: {TARGET_IMAGE_PATH}")

## Step 2: Upload or Select Product Image
You can either:
1. **Option A (Default):** Upload a file directly from your computer using the Colab file picker.
2. **Option B:** Specify a path to an existing file in `/content` or Google Drive.

In [ ]:
# Set custom_file_path if you already have an image at a specific path (e.g. from Drive or curl)
# Leave it empty ("") to open the interactive Colab upload button.
CUSTOM_FILE_PATH = ""

uploaded_source_path = None

if CUSTOM_FILE_PATH.strip() and os.path.exists(CUSTOM_FILE_PATH.strip()):
    uploaded_source_path = CUSTOM_FILE_PATH.strip()
    print(f"✅ Using existing file at: {uploaded_source_path}")
else:
    print("Please select your product image file (PNG, JPG/JPEG, or WebP) to upload:")
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]
            uploaded_source_path = os.path.join("/content", filename)
            print(f"\n✅ Uploaded successfully: {uploaded_source_path} ({len(uploaded[filename]):,} bytes)")
        else:
            print("\n⚠️ No file was selected.")
    except ImportError:
        print("Not in Google Colab environment. Please set CUSTOM_FILE_PATH to a valid local image.")

## Step 3: Image Inspection & Technical Validation
Validates file integrity, dimensions, color modes, transparency, and checks suitability for 3D reconstruction.

In [ ]:
import io
from PIL import Image

SUPPORTED_FORMATS = {"PNG", "JPEG", "JPG", "WEBP"}

validation_results = {
    "file_exists": False,
    "file_size_kb": 0.0,
    "format": None,
    "width": 0,
    "height": 0,
    "aspect_ratio": 0.0,
    "color_mode": None,
    "has_transparency": False,
    "is_valid": False,
    "warnings": []
}

if uploaded_source_path and os.path.exists(uploaded_source_path):
    validation_results["file_exists"] = True
    file_size_bytes = os.path.getsize(uploaded_source_path)
    validation_results["file_size_kb"] = file_size_bytes / 1024.0
    
    try:
        with Image.open(uploaded_source_path) as img:
            img_format = img.format.upper() if img.format else "UNKNOWN"
            validation_results["format"] = img_format
            validation_results["width"], validation_results["height"] = img.size
            validation_results["color_mode"] = img.mode
            
            if validation_results["height"] > 0:
                validation_results["aspect_ratio"] = round(img.width / img.height, 3)
            
            # Check transparency
            if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
                validation_results["has_transparency"] = True
            
            # Check format support
            if img_format in SUPPORTED_FORMATS:
                validation_results["is_valid"] = True
            else:
                validation_results["warnings"].append(f"Format '{img_format}' is not in supported list ({SUPPORTED_FORMATS}).")
                
            # Dimension checks for 3D reconstruction quality
            if img.width < 256 or img.height < 256:
                validation_results["warnings"].append("Image resolution is very low (<256px). TripoSR reconstruction quality will be poor.")
            elif img.width < 512 or img.height < 512:
                validation_results["warnings"].append("Image resolution is modest (<512px). Recommended resolution is >=512x512.")
            
            # Aspect ratio warning (extreme panoramic)
            ar = validation_results["aspect_ratio"]
            if ar < 0.5 or ar > 2.0:
                validation_results["warnings"].append(f"Aspect ratio ({ar}:1) is extreme. Square or near-square (1:1 to 4:3) works best for 3D reconstruction.")
                
    except Exception as e:
        validation_results["warnings"].append(f"Failed to decode image: {str(e)}")
else:
    validation_results["warnings"].append("No uploaded file found to validate.")

print("=" * 72)
print("LAZYAD — SPIKE S2 IMAGE VALIDATION REPORT")
print("=" * 72)
print(f"• Source File Path     : {uploaded_source_path}")
print(f"• File Size            : {validation_results['file_size_kb']:.2f} KB")
print(f"• Image Format         : {validation_results['format']}")
print(f"• Dimensions (WxH)     : {validation_results['width']} x {validation_results['height']} px")
print(f"• Aspect Ratio         : {validation_results['aspect_ratio']}:1")
print(f"• Color Mode           : {validation_results['color_mode']}")
print(f"• Contains Alpha/Trans.: {validation_results['has_transparency']}")
print(f"• Decodable & Valid    : {validation_results['is_valid']}")
if validation_results["warnings"]:
    print("\n⚠️ Advisories / Warnings:")
    for w in validation_results["warnings"]:
        print(f"  - {w}")
print("=" * 72)

## Step 4: Image Preparation & Standardization
Prepares the image for Spike S3 (TripoSR):
- Preserves the original file unmodified in `/content/lazyad_assets/original_<filename>`.
- Standardizes the input as a clean, high-quality PNG at `/content/lazyad_assets/product_input.png`.
- Performs **zero destructive crops or AI hallucinations**—the geometry and appearance of your product remain 100% faithful.

In [ ]:
preparation_success = False

if validation_results["is_valid"] and uploaded_source_path:
    # 1. Back up original file
    original_backup_name = "original_" + os.path.basename(uploaded_source_path)
    original_backup_path = os.path.join(ASSET_DIR, original_backup_name)
    shutil.copy2(uploaded_source_path, original_backup_path)
    
    # 2. Standardize to target PNG
    with Image.open(uploaded_source_path) as img:
        # Handle non-standard color modes (e.g. CMYK, Palette)
        if img.mode in ("RGBA", "LA"):
            standardized_img = img.copy()
        elif img.mode == "RGB":
            standardized_img = img.copy()
        else:
            # Convert palette or CMYK to RGB
            standardized_img = img.convert("RGB")
            
        # Save standardized PNG
        standardized_img.save(TARGET_IMAGE_PATH, format="PNG", optimize=False)
        
    if os.path.exists(TARGET_IMAGE_PATH) and os.path.getsize(TARGET_IMAGE_PATH) > 0:
        preparation_success = True
        target_size_kb = os.path.getsize(TARGET_IMAGE_PATH) / 1024.0
        print(f"✅ Original preserved at  : {original_backup_path}")
        print(f"✅ Standardized input saved: {TARGET_IMAGE_PATH} ({target_size_kb:.2f} KB)")
    else:
        print("❌ Failed to save standardized image.")
else:
    print("❌ Cannot prepare image because validation failed or no file was uploaded.")

## Step 5: Inline Image Preview & Visual Inspection

In [ ]:
from IPython.display import display, Image as IPImage

if preparation_success and os.path.exists(TARGET_IMAGE_PATH):
    print("\nPrepared Product Image for Spike S3 (TripoSR):")
    display(IPImage(filename=TARGET_IMAGE_PATH))
else:
    print("❌ No prepared image available to preview.")

## Step 6: Evaluated Pass / Fail Verdict (S2 Acceptance Criteria)

### Pass Criteria:
1. A supported image file (PNG, JPG/JPEG, WebP) was successfully loaded.
2. The image was decodable and its dimensions, format, and size were logged.
3. The standardized image was written to `/content/lazyad_assets/product_input.png`.
4. The saved image can be reopened and displayed inline.
5. The asset is staged and ready for Spike S3 (TripoSR 3D reconstruction).

In [ ]:
print("=" * 72)
print("SPIKE S2: EVALUATION & ACCEPTANCE VERDICT")
print("=" * 72)

c1_loaded = validation_results["file_exists"]
c2_valid = validation_results["is_valid"]
c3_saved = preparation_success and os.path.exists(TARGET_IMAGE_PATH)
c4_reopenable = False

if c3_saved:
    try:
        with Image.open(TARGET_IMAGE_PATH) as test_img:
            test_img.verify()
        c4_reopenable = True
    except Exception:
        c4_reopenable = False

print(f"1. Image File Loaded Successfully  : [{'PASS' if c1_loaded else 'FAIL'}]")
print(f"2. Valid & Supported Format Decoded : [{'PASS' if c2_valid else 'FAIL'}] ({validation_results['format']})")
print(f"3. Saved to Target Path on Disk     : [{'PASS' if c3_saved else 'FAIL'}] ({TARGET_IMAGE_PATH})")
print(f"4. Reopened and Verified Non-empty  : [{'PASS' if c4_reopenable else 'FAIL'}]")

s2_verdict = c1_loaded and c2_valid and c3_saved and c4_reopenable

print("-" * 72)
if s2_verdict:
    print("🎯 FINAL VERDICT FOR SPIKE S2: [PASS]")
    print(f"   Ready for Spike S3 (TripoSR Single-Image 3D Reconstruction).")
    print(f"   Input File: {TARGET_IMAGE_PATH} ({validation_results['width']}x{validation_results['height']} px)")
    print("\n🔎 VISUAL INSPECTION CHECKLIST FOR S3 READINESS:")
    print("   [ ] Is the watch centered and clearly visible?")
    print("   [ ] Is the background relatively clean or separated from the watch case?")
    print("   [ ] Are the dial, bezel, and strap distinguishable?")
    print("\n👉 Please share your validation output to confirm readiness for Spike S3!")
else:
    print("❌ FINAL VERDICT FOR SPIKE S2: [FAIL]")
    if not c1_loaded:
        print("   Reason: No file was uploaded or found at the specified path.")
    elif not c2_valid:
        print("   Reason: File format or decoding check failed.")
    elif not c3_saved:
        print("   Reason: Target file could not be written to disk.")
print("=" * 72)
